## WordNet Lemma Exploration

This notebook provides a quick reference for navigating the WordNet lemma data used throughout the project.

The project uses `nltk.corpus.wordnet2021` extended with the Open Multilingual Wordnet (`add_exomw()`), which adds lemmas in ~170 languages beyond English. Two interfaces are available:

- **Raw NLTK API** — direct access to synsets/lemmas for arbitrary queries.
- **`MultiLingualWordNetSynsets`** — the project wrapper that filters by supported language set, generates spacing variations, and tokenizes everything into a cuDF DataFrame.

In [1]:
import pandas as pd
from collections import Counter
from nltk.corpus import wordnet2021 as wn

from frames.representations import FrameUnembeddingRepresentation
from frames.nlp.synsets import SupportedLanguages


wn.add_exomw()

### 1. Languages available in the extended WordNet

In [2]:
langs = sorted(wn.langs())
print(f"{len(langs)} languages available")

1222 languages available


In [3]:
# I want to use only english for now, so let's filter the languages to only include English
english_langs = [lang for lang in langs if lang.startswith('en')]
english_langs

['enf_wikt', 'eng', 'eng_cldr', 'eng_wikt', 'enm_wikt']

### 2. Lemma counts per language

Counts how many (synset, lemma) pairs exist for each language.

In [4]:
lang_counts = Counter()
for synset in wn.all_synsets():
    for lang in english_langs:
        lang_counts[lang] += len(synset.lemmas(lang))

df_langs = (
    pd.Series(lang_counts, name="lemma count")
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "lang"})
)
df_langs.head(20)

OSError: Zipfile '/home/rodrigo/nltk_data/corpora/omw-1.4.zip' does not contain 'omw-1.4/wikt/wn-wikt-enf.tab'

### 3. Looking up a word

Each WordNet entry is a **synset** (a sense of a word). A synset has:
- a name like `dog.n.01` (word, POS, sense index)
- a definition and example sentences
- a list of **lemmas** — the surface forms in each language

In [10]:
WORD = "rule"

for s in wn.synsets(WORD):
    print(f"{s.name():30s}  pos={s.pos()}  '{s.definition()}'")
    eng_lemmas = [l.name() for l in s.lemmas("eng")]
    print(f"  English lemmas: {eng_lemmas}")

rule.n.01                       pos=n  'a principle or condition that customarily governs behavior'
  English lemmas: ['rule', 'regulation']
convention.n.02                 pos=n  'something regarded as a normative example'
  English lemmas: ['convention', 'normal', 'pattern', 'rule', 'formula']
rule.n.03                       pos=n  'prescribed guide for conduct or action'
  English lemmas: ['rule', 'prescript']
rule.n.04                       pos=n  '(linguistics) a rule describing (or prescribing) a linguistic practice'
  English lemmas: ['rule', 'linguistic_rule']
principle.n.01                  pos=n  'a basic generalization that is accepted as true and that can be used as a basis for reasoning or conduct'
  English lemmas: ['principle', 'rule']
rule.n.06                       pos=n  'the duration of a monarch's or government's power'
  English lemmas: ['rule']
dominion.n.01                   pos=n  'dominance or power through legal authority'
  English lemmas: ['dominion', 'rule'

### 5. Antonyms

Antonym pairs are the basis of concept-guided generation (e.g. `woman.n.01 - man.n.01`).

In [25]:
SYNSET_NAME = "man.n.01"

s = wn.synset(SYNSET_NAME)
for lemma in s.lemmas("eng"):
    antonyms = [a.name() for a in lemma.antonyms()]
    print(f"{lemma.name():20s} -> antonyms: {antonyms}")

man                  -> antonyms: ['woman']
adult_male           -> antonyms: []


In [33]:
# list 10 lemmas with antonyms
lemmas_with_antonyms = []
for s in wn.all_synsets():
    for lemma in s.lemmas("eng"):
        if lemma.antonyms() :
            lemmas_with_antonyms.append(lemma)
len(lemmas_with_antonyms)

7786